# <span style="color: #1F1DB5;">SPRINT 15 – Métodos Numéricos

# <span style="color: #1F1DB5;">Notebook 2 – Gradient Descent Avanzado: Learning Rate, Momentum y Adam

## Ficha de la sesión

| Campo | Detalle |
|---|---|
| Programa | Data Science |
| Sprint / sesión | Sprint 15 · Clase 2 |
| Tema | Gradient Descent Avanzado |
| Audiencia | Tutor |
| Tipo de sesión | teórico-práctica |
| Duración de referencia | 120 minutos, incluida práctica, validación y cierre |
| Última actualización | 2026-08-19 |



## Prerrequisitos y preparación

- **Conceptuales:** Álgebra lineal, función de costo, descenso de gradiente y Python con NumPy.
- **Técnicos:** Python 3.11 o entorno compatible; instalar dependencias desde `../../requirements.txt`.
- **Archivos y red:** usa solo rutas relativas. Cuando haya una operación externa, revisa su nota de contingencia antes de clase.
- **Reproducibilidad:** usa la semilla de referencia `42` en cualquier operación aleatoria y documenta excepciones.


## Resultado esperado y evidencia

**Entregable:** Comparación de estrategias de optimización mediante curvas de convergencia.

**Criterio de aceptación principal:** Fija semilla 42 y compara learning rates con el mismo punto inicial y criterio de parada.

La evidencia debe quedar dentro del notebook: código o procedimiento, comprobación del resultado y una interpretación breve.


## Ruta de trabajo e instrucciones de uso

1. Lee la pregunta guía, los objetivos y el entregable antes de ejecutar código.
2. Ejecuta las celdas en orden; si cambias una definición, reinicia el kernel y usa **Run All**.
3. En cada actividad sigue: **propósito → intento → validación → interpretación**.
4. Ejecuta primero en kernel limpio; guía la discusión antes de mostrar soluciones o criterios esperados.
5. Registra cualquier supuesto, fuente externa o cambio de datos.

> **Control obligatorio de esta clase:** Fija semilla 42 y compara learning rates con el mismo punto inicial y criterio de parada.


## <span style="color: #2826DE;">Objetivos de Aprendizaje

- Comprender el efecto del **learning rate** en la convergencia del modelo.
- Diferenciar entre **SGD, mini-batch y batch** gradient descent.
- Aplicar **momentum** y **Adam** como optimizadores avanzados.
- Entender por qué el **feature scaling** es crucial para gradient descent.
- Visualizar la convergencia con **diferentes configuraciones**.

### <span style="color: #2772F2;">Agenda (2 horas)

| Tema | Duración |
|---|---|
Introducción + Pregunta guía | 10 min |
Learning rate: demasiado grande vs pequeño | 20 min |
SGD vs Mini-batch vs Batch | 20 min |
Momentum y Adam | 25 min |
Feature scaling para convergencia | 15 min |
Debate: ¿SGD o Adam? | 15 min |
Cierre y Kahoot | 5 min |

## <span style="color: #2826DE;">❓ Pregunta Guía

### ¿Cómo hacer que un modelo aprenda más rápido sin perder precisión?

Gradient descent es el corazón de cómo los modelos "aprenden". Pero su velocidad y estabilidad dependen de decisiones clave: ¿qué tan grandes son los pasos? ¿usamos todos los datos o solo una muestra? ¿cómo evitamos quedarnos atrapados en mínimos locales? Hoy exploraremos los trucos que los frameworks modernos usan para resolver estos problemas.

## <span style="color: #2826DE;">Learning Rate: El Parámetro Más Importante (20 mins)

El **learning rate (α)** controla el tamaño de cada paso que da gradient descent:

`θ_nuevo = θ_actual - α × ∇L(θ)`

Donde `∇L(θ)` es el gradiente (la dirección de mayor aumento de la pérdida).

**¿Qué pasa con diferentes valores de α?**

| Learning Rate | Comportamiento |
|---|---|
| Muy grande (ej. 1.0) | El modelo "salta" demasiado, diverge, la pérdida SUBE |
| Grande (ej. 0.1) | Convergencia rápida pero puede oscilar alrededor del mínimo |
| Bueno (ej. 0.01) | Convergencia estable y suave al mínimo |
| Muy pequeño (ej. 0.0001) | Convergencia extremadamente lenta, puede no llegar nunca |

**Analogía:** Imagina que bajas una montaña con los ojos vendados. El gradiente te dice la dirección más empinada. El learning rate es el tamaño de tus pasos:
- Pasos enormes → te pasas del valle y terminas en la montaña de enfrente.
- Pasos diminutos → llegas pero te tarda toda la vida.
- Pasos adecuados → llegas al fondo eficientemente.

Implementemos gradient descent manualmente para una función cuadrática simple y veamos cómo el learning rate afecta la convergencia.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Función objetivo: f(x) = x² + 2x + 1 (mínimo en x = -1)
def f(x):
    return x**2 + 2*x + 1

def grad_f(x):
    return 2*x + 2

# Gradient descent con diferentes learning rates
learning_rates = [0.01, 0.1, 0.5, 1.05]
colors = ['blue', 'green', 'orange', 'red']
x_start = 4.0

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Plot 1: Trayectorias en la función
x_range = np.linspace(-5, 5, 100)
axes[0].plot(x_range, f(x_range), 'k-', linewidth=2, label='f(x) = x² + 2x + 1')
axes[0].axvline(x=-1, color='gray', linestyle='--', alpha=0.5, label='Mínimo (x=-1)')

for lr, color in zip(learning_rates, colors):
    x = x_start
    path = [x]
    for _ in range(25):
        x = x - lr * grad_f(x)
        path.append(x)
        if abs(x) > 100:  # divergió
            break
    path = np.array(path)
    valid = np.abs(path) < 10
    axes[0].plot(path[valid], f(path[valid]), 'o-', color=color, markersize=4,
                 alpha=0.7, label=f'α={lr}')

axes[0].set_xlabel('x'); axes[0].set_ylabel('f(x)')
axes[0].set_title('Trayectoria del Gradient Descent', fontweight='bold')
axes[0].legend(); axes[0].set_ylim(-1, 30); axes[0].grid(True, alpha=0.3)

# Plot 2: Loss vs iteración
for lr, color in zip(learning_rates, colors):
    x = x_start
    losses = [f(x)]
    for _ in range(25):
        x = x - lr * grad_f(x)
        losses.append(f(x))
        if f(x) > 1000:
            break
    axes[1].plot(losses, 'o-', color=color, label=f'α={lr}', markersize=4)

axes[1].set_xlabel('Iteración'); axes[1].set_ylabel('Loss f(x)')
axes[1].set_title('Convergencia de la Pérdida', fontweight='bold')
axes[1].legend(); axes[1].set_ylim(-1, 35); axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("💡 Observa:")
print("   α=0.01: converge pero MUY lento")
print("   α=0.1:  converge rápido y estable ✅")
print("   α=0.5:  converge pero oscila")
print("   α=1.05: ¡DIVERGE! La pérdida sube en vez de bajar")

Preguntas:

- ¿Por qué un learning rate de 1.05 hace que la pérdida suba en vez de bajar?

- Si no sabes qué learning rate usar, ¿qué estrategia seguirías para encontrar uno bueno?

- ¿Hay alguna forma de que el learning rate cambie durante el entrenamiento? (Pista: scheduling)

### Learning Rate Scheduling: Lo mejor de ambos mundos

En la práctica, el learning rate no tiene que ser fijo. Las estrategias de **scheduling** cambian el LR durante el entrenamiento:

| Estrategia | Comportamiento | Cuándo usar |
|---|---|---|
| Step Decay | Reduce LR cada N épocas (×0.1) | Entrenamiento largo |
| Cosine Annealing | LR sigue curva coseno: alto→bajo→alto | Estado del arte |
| Warmup + Decay | Empieza bajo, sube rápido, baja gradual | Transformers/NLP |
| ReduceOnPlateau | Reduce LR si loss no mejora en N épocas | Entrenamiento adaptativo |

La idea es: pasos grandes al inicio (explorar) y pasos pequeños al final (ajuste fino).
Esto es lo que hacen PyTorch y TensorFlow en la práctica con `lr_scheduler`.

## <span style="color: #2826DE;">SGD vs Mini-batch vs Batch Gradient Descent (20 mins)

Cuando tenemos millones de datos, calcular el gradiente con TODOS los datos en cada paso es costoso. Existen tres variantes:

**Batch Gradient Descent (BGD):**
- Usa **todos** los datos para calcular el gradiente.
- Gradiente exacto → convergencia suave.
- Muy lento con datasets grandes (cálculo en cada paso = O(n)).

**Stochastic Gradient Descent (SGD):**
- Usa **un solo dato** aleatorio por paso.
- Gradiente muy ruidoso → convergencia errática pero rápida.
- Puede escapar de mínimos locales (el ruido ayuda).
- Cálculo barato → muchos pasos por segundo.

**Mini-batch Gradient Descent:**
- Usa un **subconjunto** aleatorio (batch_size = 32, 64, 128...).
- **Compromiso ideal**: menos ruido que SGD, más rápido que batch.
- Es lo que se usa en la práctica en deep learning y ML moderno.

**Analogía:** Imagina encuestar a la población de México:
- Batch: preguntas a los 130 millones (exacto pero imposible).
- SGD: preguntas a 1 persona (rapidísimo pero muy impreciso).
- Mini-batch: preguntas a 1000 personas (rápido y razonablemente preciso).

Implementemos los tres tipos y comparemos cómo convergen en un problema de regresión lineal. Veremos la diferencia en estabilidad vs velocidad.

In [ ]:
# Comparación: Batch vs SGD vs Mini-batch
np.random.seed(42)

# Datos de regresión lineal: y = 3x + 2 + ruido
n_data = 500
X_data = np.random.uniform(-3, 3, n_data)
y_data = 3 * X_data + 2 + np.random.normal(0, 1, n_data)

# Función de pérdida MSE y su gradiente
def mse_grad(w, b, X, y):
    """Gradiente de MSE respecto a w y b"""
    n = len(X)
    y_pred = w * X + b
    dw = -2/n * np.sum(X * (y - y_pred))
    db = -2/n * np.sum(y - y_pred)
    return dw, db

def mse_loss(w, b, X, y):
    return np.mean((y - (w * X + b))**2)

# Implementar los tres métodos
def train_gd(method, X, y, lr=0.05, epochs=50, batch_size=32):
    w, b = 0.0, 0.0
    losses = []
    for epoch in range(epochs):
        if method == 'batch':
            dw, db = mse_grad(w, b, X, y)
            w -= lr * dw
            b -= lr * db
        elif method == 'sgd':
            idx = np.random.randint(0, len(X))
            dw, db = mse_grad(w, b, X[idx:idx+1], y[idx:idx+1])
            w -= lr * dw
            b -= lr * db
        elif method == 'mini-batch':
            indices = np.random.choice(len(X), batch_size, replace=False)
            dw, db = mse_grad(w, b, X[indices], y[indices])
            w -= lr * dw
            b -= lr * db
        losses.append(mse_loss(w, b, X, y))
    return losses, w, b

# Entrenar con cada método
losses_batch, w_b, b_b = train_gd('batch', X_data, y_data, lr=0.05, epochs=100)
losses_sgd, w_s, b_s = train_gd('sgd', X_data, y_data, lr=0.05, epochs=100)
losses_mb, w_m, b_m = train_gd('mini-batch', X_data, y_data, lr=0.05, epochs=100)

# Visualizar convergencia
plt.figure(figsize=(10, 5))
plt.plot(losses_batch, 'b-', linewidth=2, label='Batch (todos los datos)')
plt.plot(losses_sgd, 'r-', alpha=0.7, label='SGD (1 dato)')
plt.plot(losses_mb, 'g-', linewidth=1.5, label='Mini-batch (32 datos)')
plt.xlabel('Época'); plt.ylabel('MSE Loss')
plt.title('Convergencia: Batch vs SGD vs Mini-batch', fontweight='bold')
plt.legend(); plt.grid(True, alpha=0.3)
plt.show()

print(f"Parámetros reales: w=3.0, b=2.0")
print(f"Batch:      w={w_b:.3f}, b={b_b:.3f}")
print(f"SGD:        w={w_s:.3f}, b={b_s:.3f}")
print(f"Mini-batch: w={w_m:.3f}, b={b_m:.3f}")

Preguntas:

- ¿Por qué la curva de SGD es tan "ruidosa" comparada con batch?

- ¿Cuál método usarías con 10 millones de filas? ¿Por qué?

- ¿El "ruido" de SGD puede ser una ventaja? (Pista: mínimos locales)

## <span style="color: #2826DE;">Momentum y Adam: Optimizadores Inteligentes (25 mins)

El SGD básico tiene problemas:
1. Se atasca en zonas planas (gradiente pequeño → pasos minúsculos).
2. Oscila en valles estrechos (zigzaguea de pared a pared).
3. Sensible al learning rate.

**Momentum** y **Adam** resuelven estos problemas:

### Momentum: La bola que rueda

**Intuición:** Imagina una bola rodando por una colina. No se detiene inmediatamente cuando el terreno se aplana — tiene **inercia**. Momentum hace lo mismo con gradient descent.

**Fórmula:**
- `v_t = β × v_{t-1} + (1-β) × ∇L(θ)` (velocidad acumulada)
- `θ = θ - α × v_t`

Donde β ≈ 0.9 (cuánta "memoria" del gradiente pasado conservamos).

**Ventajas:** Cruza zonas planas más rápido, reduce oscilaciones en valles estrechos.

### Adam: Lo mejor de dos mundos

**Adam** (Adaptive Moment Estimation) combina:
1. **Momentum** (promedio exponencial del gradiente).
2. **Adaptive learning rate** (diferente learning rate por parámetro).

Cada parámetro tiene su propio learning rate que se adapta según la historia de sus gradientes. Features que raramente se actualizan reciben pasos más grandes.

**Es el optimizador por defecto** en deep learning y funciona bien "out of the box" sin mucho tuning.

Implementemos momentum y comparémoslo con SGD vanilla en una superficie 2D con un valle estrecho (donde SGD zigzaguea).

In [ ]:
# Comparación: SGD vs Momentum vs Adam en superficie 2D
# Función: f(x,y) = 10x² + y² (valle estrecho en dirección x)
def f_2d(x, y):
    return 10 * x**2 + y**2

def grad_2d(x, y):
    return np.array([20*x, 2*y])

# SGD vanilla
def sgd_path(lr=0.05, steps=50):
    pos = np.array([3.0, 3.0])
    path = [pos.copy()]
    for _ in range(steps):
        g = grad_2d(*pos)
        pos = pos - lr * g
        path.append(pos.copy())
    return np.array(path)

# SGD + Momentum
def momentum_path(lr=0.05, beta=0.9, steps=50):
    pos = np.array([3.0, 3.0])
    v = np.array([0.0, 0.0])
    path = [pos.copy()]
    for _ in range(steps):
        g = grad_2d(*pos)
        v = beta * v + (1 - beta) * g
        pos = pos - lr * v
        path.append(pos.copy())
    return np.array(path)

# Adam
def adam_path(lr=0.1, beta1=0.9, beta2=0.999, eps=1e-8, steps=50):
    pos = np.array([3.0, 3.0])
    m = np.array([0.0, 0.0])
    v = np.array([0.0, 0.0])
    path = [pos.copy()]
    for t in range(1, steps + 1):
        g = grad_2d(*pos)
        m = beta1 * m + (1 - beta1) * g
        v_sq = beta2 * v + (1 - beta2) * g**2
        m_hat = m / (1 - beta1**t)
        v_hat = v_sq / (1 - beta2**t)
        pos = pos - lr * m_hat / (np.sqrt(v_hat) + eps)
        v = v_sq
        path.append(pos.copy())
    return np.array(path)

# Calcular trayectorias
path_sgd = sgd_path(lr=0.045)
path_mom = momentum_path(lr=0.045, beta=0.9)
path_adam = adam_path(lr=0.3)

# Visualizar
fig, ax = plt.subplots(figsize=(10, 7))

# Contornos de la función
x_range = np.linspace(-4, 4, 100)
y_range = np.linspace(-4, 4, 100)
X_grid, Y_grid = np.meshgrid(x_range, y_range)
Z = f_2d(X_grid, Y_grid)
ax.contour(X_grid, Y_grid, Z, levels=30, cmap='viridis', alpha=0.6)

# Trayectorias
ax.plot(path_sgd[:, 0], path_sgd[:, 1], 'ro-', markersize=3, linewidth=1.5, label='SGD vanilla', alpha=0.8)
ax.plot(path_mom[:, 0], path_mom[:, 1], 'bs-', markersize=3, linewidth=1.5, label='SGD + Momentum', alpha=0.8)
ax.plot(path_adam[:, 0], path_adam[:, 1], 'g^-', markersize=3, linewidth=1.5, label='Adam', alpha=0.8)
ax.plot(0, 0, 'k*', markersize=15, label='Mínimo (0,0)')

ax.set_xlabel('x'); ax.set_ylabel('y')
ax.set_title('Trayectoria de Optimizadores: f(x,y) = 10x² + y²\n(Valle estrecho en dirección x)', fontweight='bold')
ax.legend(fontsize=11); ax.grid(True, alpha=0.3)
ax.set_xlim(-4, 4); ax.set_ylim(-4, 4)
plt.tight_layout()
plt.show()

print("💡 Observa cómo:")
print("   SGD zigzaguea de pared a pared del valle")
print("   Momentum suaviza las oscilaciones y avanza más directo")
print("   Adam se adapta a cada dimensión y converge rápido")

Preguntas:

- ¿Por qué SGD zigzaguea en el valle estrecho? (Pista: el gradiente en x es 10× más fuerte que en y)

- ¿Cómo ayuda momentum a reducir las oscilaciones?

- ¿En qué caso SGD puro podría ser MEJOR que Adam?

### Tabla Comparativa de Optimizadores

| Optimizador | Fórmula clave | Hiperparámetros | Ventaja |
|---|---|---|---|
| SGD | θ -= α×∇L | α (learning rate) | Simple, buen baseline |
| SGD + Momentum | v = βv + ∇L; θ -= αv | α, β≈0.9 | Cruza zonas planas, menos oscilación |
| RMSProp | Adapta LR por parámetro | α, decay≈0.9 | Bueno para RNN |
| Adam | Momentum + adaptive LR | α≈0.001, β1≈0.9, β2≈0.999 | Funciona bien "out of the box" |
| AdamW | Adam + weight decay | α, β1, β2, wd | Estado del arte para Transformers |

En la mayoría de problemas de ML clásico (sklearn), no necesitas elegir optimizador manualmente — sklearn lo maneja. Pero en deep learning (PyTorch, TensorFlow), **esta decisión es crucial**.

## <span style="color: #2826DE;">Feature Scaling para Convergencia (15 mins)

El **feature scaling** (normalización o estandarización) no solo es bueno para los modelos — es **esencial** para que gradient descent converja correctamente.

**¿Por qué?**
Cuando los features tienen escalas muy diferentes (ej. edad [0-100] vs ingreso [0-1000000]):
- La superficie de pérdida se vuelve un "valle muy estrecho y largo".
- El gradiente en la dimensión grande domina, causando zigzagueo.
- Necesitas un learning rate diminuto para no divergir.

**Con scaling:**
- La superficie se vuelve más "redonda" (isótropa).
- Gradient descent va directo al mínimo.
- Puedes usar un learning rate más grande.
- Converge en muchas menos iteraciones.

**Opciones:**
- `StandardScaler`: resta media, divide por std → datos con media=0, std=1.
- `MinMaxScaler`: escala a [0, 1].
- Para gradient descent, `StandardScaler` es generalmente mejor.

Demostremos visualmente la diferencia entre GD sin escalar y con escalar. El efecto es dramático.

In [ ]:
# Demostración: GD con y sin feature scaling
np.random.seed(42)

# Datos con escalas MUY diferentes
# Feature 1: rango [0, 1], Feature 2: rango [0, 1000]
n = 200
X_raw = np.column_stack([
    np.random.uniform(0, 1, n),
    np.random.uniform(0, 1000, n)
])
y_reg = 2 * X_raw[:, 0] + 0.005 * X_raw[:, 1] + np.random.normal(0, 0.5, n)

# GD para regresión lineal con 2 features
def gd_linear(X, y, lr, steps):
    w = np.zeros(X.shape[1])
    path_w = [w.copy()]
    losses = []
    for _ in range(steps):
        pred = X @ w
        error = pred - y
        grad = 2/len(y) * (X.T @ error)
        w = w - lr * grad
        path_w.append(w.copy())
        losses.append(np.mean(error**2))
    return np.array(path_w), losses

# Sin escalar
path_raw, loss_raw = gd_linear(X_raw, y_reg, lr=0.0000005, steps=100)

# Con StandardScaler
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_raw)
path_scaled, loss_scaled = gd_linear(X_scaled, y_reg, lr=0.1, steps=100)

# Visualizar convergencia
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

axes[0].plot(loss_raw, 'r-', linewidth=2)
axes[0].set_title('Sin Escalar (lr=0.0000005)', fontweight='bold')
axes[0].set_xlabel('Iteración'); axes[0].set_ylabel('MSE Loss')
axes[0].grid(True, alpha=0.3)

axes[1].plot(loss_scaled, 'g-', linewidth=2)
axes[1].set_title('Con StandardScaler (lr=0.1)', fontweight='bold')
axes[1].set_xlabel('Iteración'); axes[1].set_ylabel('MSE Loss')
axes[1].grid(True, alpha=0.3)

plt.suptitle('Impacto del Feature Scaling en Gradient Descent', fontweight='bold', fontsize=13)
plt.tight_layout()
plt.show()

print("💡 Sin escalar: necesitas lr minúsculo y aún converge lento")
print("   Con escalar: lr razonable y convergencia rápida")
print(f"\n   Loss final sin escalar: {loss_raw[-1]:.4f}")
print(f"   Loss final con escalar: {loss_scaled[-1]:.4f}")

Preguntas:

- ¿Por qué el learning rate necesario sin escalar es 200,000× más pequeño que con escalar?

- ¿StandardScaler o MinMaxScaler para gradient descent? ¿Por qué?

- ¿Qué modelos NO necesitan feature scaling? (Pista: árboles de decisión)

### Resumen: Checklist para Gradient Descent en la Práctica

Antes de entrenar cualquier modelo basado en gradient descent, sigue este checklist:

1. ✅ **Escalar features** → `StandardScaler` (media=0, std=1).
2. ✅ **Elegir optimizador** → Adam como default, SGD+momentum para producción.
3. ✅ **Learning rate inicial** → 0.001 para Adam, 0.01-0.1 para SGD.
4. ✅ **Batch size** → 32-128 para mini-batch (compromiso velocidad/estabilidad).
5. ✅ **Monitorear loss** → Si sube: reducir LR. Si no baja: aumentar LR.
6. ✅ **LR scheduling** → Reducir gradualmente para mejor convergencia final.
7. ✅ **Early stopping** → Detener si validation loss deja de mejorar.

Estos pasos aplican a logistic regression, SVM, redes neuronales, y cualquier modelo que use GD.

#### Criterios de logro

- La respuesta atiende todos los elementos de la consigna.
- El resultado incluye una comprobación observable, no solo código o una afirmación.
- La conclusión explica qué significa el resultado y menciona al menos un supuesto.
- **Control específico:** Fija semilla 42 y compara learning rates con el mismo punto inicial y criterio de parada.

> **Nota para tutor:** pide una predicción antes de ejecutar, valida el criterio en vivo y comenta al menos un error frecuente.


## <span style="color: #2826DE;">Debate: ¿SGD o Adam? ¿Cuál es mejor? (15 mins)

Vamos a debatir como comunidad de científicos de datos. Divídanse en dos equipos:

**🔵 Equipo SGD:**
Argumenten por qué SGD (con momentum) es preferible. Consideren:
- SGD generaliza mejor en la práctica (papers recientes lo demuestran).
- Con un buen learning rate schedule, SGD alcanza mejor solución final.
- SGD es más simple y tiene menos hiperparámetros.
- Los mejores modelos de visión (ResNet, etc.) se entrenan con SGD + momentum.

**🔴 Equipo Adam:**
Argumenten por qué Adam es preferible. Consideren:
- Adam converge más rápido con menos tuning.
- Adam adapta el learning rate por parámetro (útil con features sparse).
- Adam funciona "out of the box" sin necesidad de scheduling.
- Es el estándar en NLP (BERT, GPT) y en la mayoría de frameworks.

**Reglas del debate:**
1. Cada equipo tiene 3 minutos para argumentar.
2. Luego 2 minutos de contraargumentos.
3. Al final: ¿hay un ganador universal? (Spoiler: no, depende del problema).

### Resumen del debate: No hay ganador universal

| Criterio | SGD + Momentum | Adam |
|---|---|---|
| **Convergencia** | Más lenta pero mejor solución final | Más rápida, puede estancarse |
| **Tuning** | Requiere schedule de LR | Funciona bien con defaults |
| **Generalización** | Mejor en visión y tareas grandes | Puede sobreajustar |
| **Uso práctico** | Producción (cuando importa cada 0.1%) | Prototipado y NLP |
| **Sparse features** | No se adapta bien | Excelente (adaptive LR) |

**Recomendación práctica:**
- Empieza con Adam (rápido de iterar).
- Si necesitas exprimir el último 0.5% de performance, prueba SGD + cosine schedule.
- Para NLP: Adam (o variantes como AdamW).
- Para visión: SGD + momentum + schedule.

## <span style="color: #2826DE;">Tips y buenas prácticas

- **Siempre escala** tus features antes de usar gradient descent (aplica a logistic regression, SVM, redes neuronales).
- Empieza con **Adam y lr=0.001** como baseline. Ajusta después si es necesario.
- Si el loss sube: tu learning rate es demasiado grande. Redúcelo a la mitad.
- Si el loss baja muy lento: puedes aumentar el learning rate.
- **Learning rate scheduling** (reducir LR gradualmente) mejora la convergencia final.
- En sklearn, `SGDClassifier` y `SGDRegressor` usan mini-batch por defecto — son perfectos para datos grandes.
- Los árboles de decisión y Random Forest NO usan gradient descent. Por eso no necesitan scaling.

## <span style="color: #2826DE;">Cierre y Kahoot (5 mins)

Kahoot - Preguntas sugeridas

1️⃣ ¿Qué pasa si el learning rate es demasiado grande?

- El modelo converge más rápido
- El modelo diverge: la pérdida sube ✅
- No tiene ningún efecto
- El modelo hace underfitting


2️⃣ ¿Qué variante de GD usa UN solo dato por paso?

- Batch Gradient Descent
- Mini-batch Gradient Descent
- Stochastic Gradient Descent (SGD) ✅
- Adam


3️⃣ ¿Cuál es la intuición detrás de "momentum"?

- Usar gradientes de segundo orden
- Acumular velocidad como una bola rodando ✅
- Multiplicar el gradiente por un número grande
- Cambiar el learning rate en cada paso


4️⃣ ¿Por qué el feature scaling ayuda a gradient descent?

- Hace la superficie de pérdida más "redonda" y facilita la convergencia ✅
- Porque sklearn lo requiere obligatoriamente
- Porque reduce el número de features
- Porque convierte datos categóricos a numéricos


5️⃣ ¿Qué combina el optimizador Adam?

- Dropout y regularización
- Momentum y learning rate adaptativo ✅
- Batch normalization y skip connections
- SGD y árboles de decisión


6️⃣ ¿Qué modelos NO necesitan feature scaling?

- Regresión Logística
- KNN
- Árboles de Decisión y Random Forest ✅
- Redes Neuronales

## Cierre, límites y siguientes pasos

### Síntesis

La clase de **Gradient Descent Avanzado** conecta los conceptos con una evidencia reproducible. El trabajo se considera completo cuando produce **comparación de estrategias de optimización mediante curvas de convergencia.** y satisface el control técnico definido al inicio.

### Limitaciones que deben declararse

- Los resultados dependen del alcance y la calidad de los datos o ejemplos usados.
- Una ejecución correcta no demuestra por sí sola que el resultado generalice a otros contextos.
- Las decisiones de modelado, limpieza o comunicación deben justificarse según el problema.

### Lista de salida

- [ ] Respondí la pregunta guía con evidencia.
- [ ] Validé el resultado con el criterio de logro: Fija semilla 42 y compara learning rates con el mismo punto inicial y criterio de parada.
- [ ] Documenté supuestos, límites y decisiones.
- [ ] Dejé rutas portables, sin credenciales ni información privada.

Antes de publicar, usa Restart Kernel and Run All y conserva únicamente outputs breves que ayuden a verificar el resultado esperado.

### Ayuda y recursos

- Revisa los ejemplos y el glosario de esta clase antes de cambiar el enfoque.
- Lleva dudas concretas y el mensaje de error completo a los canales de soporte u office hours.
- Continúa con la siguiente clase del sprint y reutiliza este entregable como evidencia de aprendizaje.
